# 1 · Business analytics: la encuesta de la clase 1

**Pregunta:** ¿quién respondió, qué roles hay en la sala y cómo cambia la confianza según el rol?

Ejecuta `bash demos/clase-02/pipeline/etl_only.sh --dataset class1` desde la raíz del repo antes de abrir este notebook. El ETL escribe las respuestas completas y una vista agregada sin identificadores. Aquí las gráficas usan esa vista; el cruce se agrega antes de mostrar resultados y nunca dibuja `participant_key`.

Las seis preguntas y sus opciones se leen de los metadatos reales de D1. Los selectores encuentran las preguntas por texto y fallan con una instrucción clara si el wording cambia.

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

CATALOG = "workspace"
SCHEMA = "ai4data"
DIM = f"{CATALOG}.{SCHEMA}.c1_dim_participante"
FACT = f"{CATALOG}.{SCHEMA}.c1_fct_respuestas"
SUMMARY = f"{CATALOG}.{SCHEMA}.c1_vw_resumen_respuestas"

def require_table(full_name):
    catalog, schema, table = full_name.split(".")
    names = {r.tableName for r in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()}
    if table not in names:
        raise RuntimeError(
            f"Falta {full_name}. Ejecuta desde el repo: "
            "bash demos/clase-02/pipeline/etl_only.sh --dataset class1"
        )

for table in (DIM, FACT, SUMMARY):
    require_table(table)

dim = spark.table(DIM)
fact = spark.table(FACT)
summary = spark.table(SUMMARY)
n_respondents = dim.count()
print(f"Personas únicas en la clase 1: {n_respondents:,}")

## Cobertura de respuestas

Primero contamos participantes por pregunta y usamos el total del grupo como denominador de cobertura. La barra representa respuestas recibidas; el `n` encima de cada barra hace visible el tamaño.

In [ ]:
coverage = (
    summary.groupBy("poll_code", "pregunta")
    .agg(F.max("denominador_pregunta").alias("n_respuestas"))
    .orderBy("poll_code")
)
coverage_pd = coverage.toPandas()
coverage_pd["cobertura_pct"] = 100 * coverage_pd["n_respuestas"] / n_respondents
display(coverage_pd[["pregunta", "n_respuestas", "cobertura_pct"]].round({"cobertura_pct": 1}))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(coverage_pd["pregunta"], coverage_pd["n_respuestas"], color="#6c5ce7")
ax.invert_yaxis()
ax.set_xlabel("Respuestas")
ax.set_title("Cobertura de las preguntas de la clase 1")
ax.set_xlim(0, max(n_respondents, int(coverage_pd["n_respuestas"].max())) * 1.12)
for bar, n in zip(bars, coverage_pd["n_respuestas"]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, f"{int(n)}", va="center")
fig.tight_layout()
display(fig)
plt.close(fig)

## Distribución de roles declarados

La pregunta y las categorías vienen de D1. Solo representamos la vista agregada.

In [ ]:
questions = [r.pregunta for r in fact.select("pregunta").distinct().collect()]

def find_question(fragment):
    matches = [q for q in questions if fragment.casefold() in q.casefold()]
    if len(matches) != 1:
        raise ValueError(
            f"Esperaba una pregunta que contenga {fragment!r}; encontré {len(matches)}. "
            f"Preguntas disponibles: {questions}"
        )
    return matches[0]

role_q = find_question("trabajo principal")
role_pd = (
    summary.where(F.col("pregunta") == role_q)
    .select("respuesta", "n_respuestas", "porcentaje")
    .orderBy(F.desc("n_respuestas"))
    .toPandas()
)
display(role_pd)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(role_pd["respuesta"], role_pd["n_respuestas"], color="#00b894")
ax.set_ylabel("Personas")
ax.set_title("Trabajo principal declarado")
ax.tick_params(axis="x", labelrotation=25)
for bar, pct in zip(bars, role_pd["porcentaje"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{pct:.1f}%", ha="center", fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

## ¿Cambia la confianza por rol?

Unimos tres respuestas de la misma persona dentro del cómputo distribuido y agregamos antes de traer nada al notebook. La tabla y las gráficas solo contienen grupos: cada `n` cuenta personas con las tres respuestas.

In [ ]:
ai_use_q = find_question("últimos 7 días")
trust_q = find_question("en un análisis generado por IA")
role_q = find_question("trabajo principal")

selected = fact.where(F.col("pregunta").isin(role_q, ai_use_q, trust_q))
by_person = selected.groupBy("participant_key").agg(
    F.max(F.when(F.col("pregunta") == role_q, F.col("respuesta"))).alias("rol"),
    F.max(F.when(F.col("pregunta") == ai_use_q, F.col("respuesta"))).alias("uso_ia_7d"),
    F.max(F.when(F.col("pregunta") == trust_q, F.col("respuesta").cast("double"))).alias("confianza"),
)
complete = by_person.where(
    F.col("rol").isNotNull() & F.col("uso_ia_7d").isNotNull() & F.col("confianza").isNotNull()
)
by_role = (
    complete.groupBy("rol")
    .agg(
        F.count("*").alias("n"),
        F.round(100 * F.avg(F.when(F.lower("uso_ia_7d") == "sí", 1.0).otherwise(0.0)), 1)
         .alias("uso_ia_7d_pct"),
        F.round(F.avg("confianza"), 2).alias("confianza_media"),
    )
    .orderBy(F.desc("n"))
)
by_role_pd = by_role.toPandas()
display(by_role_pd)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(by_role_pd["rol"], by_role_pd["uso_ia_7d_pct"], color="#fdcb6e")
axes[0].set_title("Usó IA en los últimos 7 días")
axes[0].set_ylabel("Personas (%)")
axes[0].set_ylim(0, 100)
axes[1].bar(by_role_pd["rol"], by_role_pd["confianza_media"], color="#74b9ff")
axes[1].set_title("Confianza en análisis generado por IA")
axes[1].set_ylabel("Promedio (1–5)")
axes[1].set_ylim(0, 5)
for ax in axes:
    ax.tick_params(axis="x", labelrotation=25)
    for label in ax.get_xticklabels():
        label.set_ha("right")
fig.suptitle("Misma cohorte con rol, uso reciente y confianza · n por rol en la tabla")
fig.tight_layout()
display(fig)
plt.close(fig)
print("Listo. Casi: buenas métricas requieren definiciones acordadas y capacidad de actuar.")
dbutils.notebook.exit(
    f"C1_BUSINESS_COMPLETE|respondents={n_respondents}|questions={len(coverage_pd)}|"
    f"roles={len(role_pd)}|cross_tab_roles={len(by_role_pd)}"
)